# Phase 4 笔记本 2：量子计算辅助力场开发

## 与博士工作的直接衔接

博士阶段背景：
- **ReaxFF**：以量子化学训练数据标定的反应力场
- **机器学习**：ML-FF、NNP（神经网络势）
- **分子动力学**：基于上述力场的 MD 模拟

量子计算带来两类新机会：

1. **更好的训练数据**：在 CCSD(T) 过于昂贵时，用 VQE 生成 PES 数据
2. **量子机器学习**：用量子核方法更高效地学习 PES

```
经典流程：                  量子增强流程：
  DFT/CCSD(T) 数据            VQE/SQD 数据（更高精度）
      |                              |
  ML 拟合                     量子 ML / QNN 拟合
  (NNP, GAP, ReaxFF)          (QKE, 量子核岭回归)
      |                              |
  MD 模拟                     MD 模拟
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 演示：VQE 可比 DFT 更准确地生成 PES 数据
# 示例：N2 三键解离（DFT 已知失效区）

# N2 参考几何（STO-3G）
r_N2 = np.array([0.9, 1.0, 1.1, 1.2, 1.5, 2.0, 2.5, 3.0, 3.5])

# 示意势能面（Ha，相对 E_min）
# DFT (PBE)：平衡区尚可，解离区失效
E_dft = np.array([0.45, 0.08, 0.00, 0.03, 0.30, 0.95, 1.40, 1.55, 1.60])
# CCSD(T)：金标准，代价 O(N^7)
E_ccsdt = np.array([0.48, 0.10, 0.00, 0.03, 0.28, 0.80, 1.20, 1.35, 1.40])
# VQE (UCCSD / CASSCF(6,6))：在解离区捕捉强关联
E_vqe = E_ccsdt + np.random.default_rng(42).normal(0, 0.008, len(r_N2))  # 带小随机误差的 VQE
# DFT 训练数据（限制在小 R 区间，DFT 尚可靠）
r_dft_train = r_N2[:4]
E_dft_train = E_dft[:4]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(r_N2, E_ccsdt, 'r-o', ms=6, lw=2, label='CCSD(T)（金标准）')
ax.plot(r_N2, E_vqe, 'b--s', ms=6, lw=2, label='VQE-UCCSD（精度相近）')
ax.plot(r_N2, E_dft, 'g-.^', ms=5, lw=1.5, label='DFT/PBE（解离区失效）')
ax.axvline(x=1.1, color='gray', ls=':', lw=1.5, label='平衡几何')
ax.axvline(x=2.0, color='orange', ls=':', lw=1.5, label='DFT 在此之外不可靠')
ax.set_xlabel('N–N 距离 (Å)'); ax.set_ylabel('相对能量 (Ha)')
ax.set_title('N2 PES：DFT 与 VQE、CCSD(T) 对比')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(r_N2, np.abs(E_vqe - E_ccsdt)*1000, 'b-s', lw=2, label='VQE 相对 CCSD(T) 误差')
ax.plot(r_N2, np.abs(E_dft - E_ccsdt)*1000, 'g-.^', lw=1.5, label='DFT 相对 CCSD(T) 误差')
ax.axhline(y=1.6, color='r', ls='--', lw=1.5, label='化学精度 1.6 mHa')
ax.set_xlabel('N–N 距离 (Å)'); ax.set_ylabel('相对 CCSD(T) 误差 (mHa)')
ax.set_title('方法精度对比')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('N2_PES_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('要点：在解离区 VQE 可达到与 CCSD(T) 相当的精度')
print('且 VQE 为多项式标度（非 O(N^7)）→ 对大体系相对更可行')
print()
print('对 ReaxFF 训练：VQE 可为以下情形提供可靠 PES：')
print('  - 断键/成键（超出 DFT 可靠区）')
print('  - 过渡金属反应中间体（自由基态）')
print('  - 电荷转移过程')


## 2. 力场的量子机器学习

### 与你已有工作的联系
你已用 QML 构建过分子力场；本节把所需的量子核框架系统化。

### 量子核方法

力场中的经典核岭回归（KRR）：
$$E(\mathbf{R}) = \sum_i \alpha_i k(\mathbf{R}, \mathbf{R}_i)$$

其中 $k(\mathbf{R}, \mathbf{R}')$ 为核函数（相似度）。

**量子核**：用量子线路计算相似度！
$$k_Q(\mathbf{x}, \mathbf{x}') = |\langle 0 | U^\dagger(\mathbf{x}') U(\mathbf{x}) | 0 \rangle|^2$$

量子线路 $U(\mathbf{x})$ 将分子特征编码为量子态；
内积可通过测量两条线路态的重叠来估计。

**理论动机**：量子核可表达特征空间中某些在经典下指数难求的关联结构。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 用于 PES 拟合的量子核
# 特征编码：分子描述符 x → 量子线路 U(x)

def quantum_feature_map(x, n_qubits=4):
    """类 ZZFeatureMap 的量子特征编码。"""
    qc = QuantumCircuit(n_qubits)
    # 层 1：Hadamard + 单比特旋转
    qc.h(range(n_qubits))
    for i in range(n_qubits):
        qc.p(x[i % len(x)], i)
    # 层 2：ZZ 相互作用（刻画两体关联）
    for i in range(n_qubits-1):
        qc.cx(i, i+1)
        qc.p(2*(np.pi - x[i % len(x)])*(np.pi - x[(i+1) % len(x)]), i+1)
        qc.cx(i, i+1)
    return qc

def quantum_kernel(x1, x2, n_qubits=4):
    """量子核：|<psi(x1)|psi(x2)>|^2"""
    sv1 = Statevector.from_instruction(quantum_feature_map(x1, n_qubits))
    sv2 = Statevector.from_instruction(quantum_feature_map(x2, n_qubits))
    return abs(sv1.inner(sv2))**2

def rbf_kernel(x1, x2, gamma=1.0):
    """经典 RBF 核（对比用）。"""
    return np.exp(-gamma * np.linalg.norm(x1 - x2)**2)

# 合成一维 PES 训练数据（N2 键长）
# 特征：[键长] → 能量
np.random.seed(42)
r_train = np.linspace(0.8, 3.5, 30)
# 真实 PES：Morse 势（模拟从头算数据）
De, a, re = 0.5, 2.0, 1.1
E_true = lambda r: De*(1 - np.exp(-a*(r-re)))**2

E_train = E_true(r_train) + np.random.normal(0, 0.003, len(r_train))  # 加噪声

# 构造核矩阵
n = len(r_train)
X = r_train.reshape(-1, 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 计算量子核矩阵
print('正在计算量子核矩阵...')
K_quantum = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        k = quantum_kernel(X_scaled[i], X_scaled[j], n_qubits=2)
        K_quantum[i,j] = K_quantum[j,i] = k

# 计算经典 RBF 核
K_rbf = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        k = rbf_kernel(X_scaled[i], X_scaled[j], gamma=2.0)
        K_rbf[i,j] = K_rbf[j,i] = k

# 用核岭回归拟合力场
r_test = np.linspace(0.7, 4.0, 100)
X_test = scaler.transform(r_test.reshape(-1,1))

# 经典 RBF 核 KRR
krr_rbf = KernelRidge(kernel='precomputed', alpha=1e-4)
krr_rbf.fit(K_rbf, E_train)
K_rbf_test = np.array([[rbf_kernel(X_test[i], X_scaled[j], gamma=2.0) for j in range(n)] for i in range(len(r_test))])
E_pred_rbf = krr_rbf.predict(K_rbf_test)

# 量子 KRR
krr_q = KernelRidge(kernel='precomputed', alpha=1e-4)
krr_q.fit(K_quantum, E_train)
K_q_test = np.array([[quantum_kernel(X_test[i], X_scaled[j], n_qubits=2) for j in range(n)] for i in range(len(r_test))])
E_pred_q = krr_q.predict(K_q_test)

# 参考曲线
E_ref = E_true(r_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(r_train, E_train, 'ko', ms=5, label='训练数据（VQE/CCSD(T)）', zorder=5)
ax1.plot(r_test, E_ref, 'k-', lw=2, label='真实 PES（Morse）')
ax1.plot(r_test, E_pred_rbf, 'g--', lw=2, label='经典 KRR（RBF 核）')
ax1.plot(r_test, E_pred_q, 'b-.', lw=2, label='量子 KRR（QKE）')
ax1.set_xlabel('键长 (Å)'); ax1.set_ylabel('能量 (Ha)')
ax1.set_title('力场拟合：经典核与量子核')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

ax2.plot(r_test, np.abs(E_pred_rbf - E_ref)*1000, 'g--', lw=2, label='经典 KRR 误差')
ax2.plot(r_test, np.abs(E_pred_q - E_ref)*1000, 'b-.', lw=2, label='量子 KRR 误差')
ax2.axhline(y=1.6, color='r', ls='--', lw=1.5, label='化学精度')
ax2.set_xlabel('键长 (Å)'); ax2.set_ylabel('误差 (mHa)')
ax2.set_title('力场精度')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('QML_forcefield.png', dpi=150, bbox_inches='tight')
plt.show()

rmse_rbf = np.sqrt(np.mean((E_pred_rbf - E_ref)**2)) * 1000
rmse_q   = np.sqrt(np.mean((E_pred_q   - E_ref)**2)) * 1000
print(f'经典 KRR 的 RMSE: {rmse_rbf:.2f} mHa')
print(f'量子 KRR 的 RMSE:   {rmse_q:.2f} mHa')
print('说明：简单一维情形经典核往往已足够')
print('量子优势更可能出现在：高维、强关联 PES')


## 3. 面向 ReaxFF 的 VQE 训练数据

### 直接可做的研究方向

ReaxFF 以往多以 DFT 数据标定；可把训练集升级为「DFT + 量子」混合：

```python
# 当前 ReaxFF 流程：
# 1. DFT 计算（VASP/CP2K）→ 训练能量/力
# 2. ReaxFF 优化（ADF、LAMMPS）→ 参数

# 引入量子计算的增强流程：
# 1a. DFT：大部分构型（快，O(N^3)）
# 1b. VQE：断键、过渡金属反应等困难情形（更高精度）
# 2.  合并 DFT + VQE 训练集
# 3.  ReaxFF / NNP 参数化（工具不变，数据更好）
```

### 相对纯 DFT 的改进点

| 构型 | DFT 问题 | VQE 思路 |
|------|----------|----------|
| Fe–O 断键 | 自旋污染 | 精确自旋态 |
| FeO 表面氧空位 | 强关联 | CAS-VQE |
| 含自由基的过渡态 | 多参考 | UCCSD-VQE |
| 范德华结合 | DFT-D3 近似 | 接近 MP2 品质的 VQE |

## 4. Qiskit Machine Learning 与 QML 力场

官方文档：https://qiskit-community.github.io/qiskit-machine-learning/


In [ ]:
# Qiskit Machine Learning 力场拟合示例模板
# 展示量子神经网络/量子核思路

QML_TEMPLATE = '''
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVR  # 量子支持向量回归
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from sklearn.preprocessing import StandardScaler
import numpy as np

# 步骤 1：准备分子描述符
# 可用：键长、键角、对称函数（如 ACSF/SOAP）
X_train = compute_descriptors(structures_train)   # (N_train, N_features)
E_train = vqe_energies_train                       # (N_train,)

# 步骤 2：特征标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# 步骤 3：量子特征映射
n_features = X_scaled.shape[1]
feature_map = ZZFeatureMap(feature_dimension=n_features, reps=2)

# 步骤 4：量子核
qkernel = FidelityQuantumKernel(feature_map=feature_map)

# 步骤 5：核岭回归（或 QSVR）
from sklearn.kernel_ridge import KernelRidge
K_train = qkernel.evaluate(X_scaled)  # N_train × N_train 核矩阵
qkrr = KernelRidge(kernel='precomputed', alpha=1e-6)
qkrr.fit(K_train, E_train)

# 步骤 6：对新结构预测能量
X_test_scaled = scaler.transform(compute_descriptors(structures_test))
K_test = qkernel.evaluate(X_test_scaled, X_scaled)
E_pred = qkrr.predict(K_test)

# 步骤 7：计算力（预测能量对坐标的梯度）
# 可用数值梯度或经描述符雅可比的解析梯度
F_pred = -np.gradient(E_pred, ...)
'''

print('QML 力场模板（Qiskit Machine Learning）：')
print(QML_TEMPLATE)

# 小结表
print('='*70)
print('与博士背景衔接的量子计算研究机会：')
print('='*70)
opportunities = [
    ('ReaxFF 训练',      '过渡金属反应的 VQE PES',           '近期（约 1 年）',   '高'),
    ('NNP/ML-FF',        'PES 的量子核回归',                 '1–2 年',           '高'),
    ('MD 模拟',          '量子训练力场 + 经典 MD',           '2–3 年',           '很高'),
    ('主动学习',         'VQE 作为自适应采样的 oracle',       '2–3 年',           '高'),
    ('量子 MD',          '核量子效应等的量子计算',           '3–5 年',           '中'),
]
for name, desc, timeline, impact in opportunities:
    print(f'  {name:<18} {desc:<40} {timeline:<15} 影响: {impact}')


## 5. 建议的后续步骤

### 近期可做（本周）
1. 安装 Qiskit Machine Learning：`pip install qiskit-machine-learning`
2. 按官方文档跑通 ZZFeatureMap + 量子核教程
3. 从博士工作中抽取一小套 ReaxFF 训练数据
4. 分别用经典 RBF 核与量子核尝试拟合对比

### 首篇论文方向（3–6 个月）
- **题目示例**：面向铁基催化剂的量子增强力场开发
- **方法**：VQE/SQD 训练数据 + QML 核拟合
- **新意**：在 CCSD(T) 难以覆盖的区间用 VQE 补数据
- **期刊**：J. Chem. Theory Comput.、npj Computational Materials 等

### 关键资料
- Qiskit ML 文档：https://qiskit-community.github.io/qiskit-machine-learning/
- 分子 QML 综述：von Lilienfeld 等 (2020) Nat. Chem.
- 量子核优势：Liu 等 (2021) Nature Physics
- arXiv 关键词：`quantum machine learning force field`、`QML potential`

---
## 恭喜完成学习计划全部四个阶段

**个人定位小结**：
```
DFT/MD 基础  +  ReaxFF/ML 经验  +  量子计算
      = 量子–经典混合计算化学
```

多数研究者只深耕其中一块；三者交叉是你差异化的优势。

**建议优先阅读**：
- RevModPhys.92.015003.pdf（系统基础）
- acs.chemrev.8b00803.pdf（化学应用）
- IBM Quantum Learning（Qiskit 实操）
